In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data/loans_clean.csv')
print(df.shape)

In [ ]:
segment = df.groupby(['grade', 'dti_bucket']).agg(
    total_loans=('default_flag', 'count'),
    total_defaults=('default_flag', 'sum'),
    avg_loan_amnt=('loan_amnt', 'mean'),
    avg_int_rate=('int_rate', 'mean')
).reset_index()

segment['default_rate'] = segment['total_defaults'] / segment['total_loans']
segment['net_profit'] = (
    segment['avg_loan_amnt'] * (segment['avg_int_rate']/100) * segment['total_loans']
) - (
    segment['avg_loan_amnt'] * segment['total_defaults']
)

print(segment.shape)
print(segment.head())

In [ ]:
thresholds = [0.15, 0.20, 0.25, 0.28, 0.35]
results = []

for t in thresholds:
    approved = segment[segment['default_rate'] < t]
    rejected = segment[segment['default_rate'] >= t]
    
    total_profit = approved['net_profit'].sum()
    total_defaults = approved['total_defaults'].sum()
    total_loans = approved['total_loans'].sum()
    approval_rate = total_loans / segment['total_loans'].sum()
    portfolio_default_rate = total_defaults / total_loans if total_loans > 0 else 0
    
    results.append({
        'threshold': f"{int(t*100)}%",
        'approved_segments': len(approved),
        'approval_rate': round(approval_rate * 100, 2),
        'portfolio_default_rate': round(portfolio_default_rate * 100, 2),
        'net_profit': round(total_profit, 2)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string())

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(results_df['threshold'], results_df['net_profit'], 
         marker='o', color='steelblue', linewidth=2.5, markersize=8)

plt.axvline(x=0, color='red', linestyle='--', alpha=0.5, label='Optimal threshold: 15%')

for i, row in results_df.iterrows():
    plt.annotate(f"${row['net_profit']:,.0f}", 
                 xy=(i, row['net_profit']),
                 xytext=(0, 12), textcoords='offset points',
                 ha='center', fontsize=9)

plt.title('Net Portfolio Profit by Approval Threshold', fontsize=14, fontweight='bold')
plt.xlabel('Default Rate Threshold')
plt.ylabel('Net Profit (USD)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('data/profit_curve.png', dpi=150)
plt.show()

In [ ]:
print(df.groupby('grade')['int_rate'].mean().round(2))

In [ ]:
df_pricing = df.copy()

df_pricing['adjusted_int_rate'] = df_pricing.apply(
    lambda row: row['int_rate'] * 1.5 if row['dti_bucket'] == 'High' else row['int_rate'],
    axis=1
)

pricing_segment = df_pricing.groupby(['grade', 'dti_bucket']).agg(
    total_loans=('default_flag', 'count'),
    total_defaults=('default_flag', 'sum'),
    avg_loan_amnt=('loan_amnt', 'mean'),
    avg_int_rate=('adjusted_int_rate', 'mean')
).reset_index()

pricing_segment['default_rate'] = pricing_segment['total_defaults'] / pricing_segment['total_loans']

pricing_segment['net_profit_adjusted'] = (
    pricing_segment['avg_loan_amnt'] * (pricing_segment['avg_int_rate']/100) * pricing_segment['total_loans']
) - (
    pricing_segment['avg_loan_amnt'] * pricing_segment['total_defaults']
)

comparison = segment[['grade','dti_bucket','net_profit']].merge(
    pricing_segment[['grade','dti_bucket','net_profit_adjusted']],
    on=['grade','dti_bucket']
)

comparison['profit_difference'] = comparison['net_profit_adjusted'] - comparison['net_profit']

print(comparison[comparison['dti_bucket'] == 'High'].to_string())

In [ ]:
high_dti = comparison[comparison['dti_bucket'] == 'High'].copy()

x = range(len(high_dti))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar([i - width/2 for i in x], high_dti['net_profit'], 
               width, label='Standard Pricing', color='steelblue')
bars2 = ax.bar([i + width/2 for i in x], high_dti['net_profit_adjusted'], 
               width, label='Risk-Based Pricing (1.5x)', color='darkorange')

ax.axhline(y=0, color='red', linestyle='--', alpha=0.7)
ax.set_xticks(list(x))
ax.set_xticklabels(high_dti['grade'])
ax.set_title('Standard vs Risk-Based Pricing — High DTI Segments', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Grade')
ax.set_ylabel('Net Profit (USD)')
ax.legend()
plt.tight_layout()
plt.savefig('data/risk_based_pricing.png', dpi=150)
plt.show()

In [ ]:
comparison.to_csv('data/risk_pricing_comparison.csv', index=False)
results_df.to_csv('data/policy_simulation_results.csv', index=False)
print("All outputs saved.")